In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install mlflow dagshub xgboost -q

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

from xgboost import XGBClassifier

In [3]:
df = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
df_id = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

df = df.merge(df_id, how="left", on="TransactionID")

target = "isFraud"
X = df.drop(columns=[target, "TransactionID"])
y = df[target]

In [4]:
# keep only numeric-heavy fast features
num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

# fill missing quickly
X[num_cols] = X[num_cols].fillna(-999)
X[cat_cols] = X[cat_cols].fillna("missing")

In [5]:
X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
X["null_count"] = X.isnull().sum(axis=1)

/tmp/ipykernel_133/2437543061.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
/tmp/ipykernel_133/2437543061.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["null_count"] = X.isnull().sum(axis=1)


In [6]:
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X.fillna(0),
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [10]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

In [17]:
model = XGBClassifier(
    n_estimators=200,      # key speed optimization
    max_depth=5,           # keep small
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",    # VERY IMPORTANT for speed
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

In [18]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [19]:
proba = model.predict_proba(X_val)[:, 1]
pred = (proba > 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_val, proba))
print("F1:", f1_score(y_val, pred))

ROC-AUC: 0.930449360687392
F1: 0.3590702644937216


In [20]:
best_t, best_f1 = 0, 0

for t in np.arange(0.1, 0.9, 0.05):
    preds = (proba > t).astype(int)
    f1 = f1_score(y_val, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best F1:", best_f1)

Best threshold: 0.8500000000000002
Best F1: 0.6071527585250901


In [22]:
mlflow.set_experiment("FAST_XGBOOST")

with mlflow.start_run():

    mlflow.log_param("model", "XGBoost_fast")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 5)

    mlflow.log_metric("auc", roc_auc_score(y_val, proba))
    mlflow.log_metric("f1", f1_score(y_val, pred))

    mlflow.sklearn.log_model(model, "model")

2026/05/07 18:40:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 18:40:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [25]:
with mlflow.start_run(run_name="XGBoost") as run:
    
    run_id = run.info.run_id

    print("🔗 MLflow Run Link:")
    print(f"https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments/FAST_XGBOOST/runs/{run_id}")

🔗 MLflow Run Link:
https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments/FAST_XGBOOST/runs/3ec79e0fa0a948639b80828c119ec6a4


In [26]:
with mlflow.start_run() as run:

    run_id = run.info.run_id

    print("RUN ID:", run_id)
    print("OPEN HERE:")
    print("https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments")

RUN ID: d642410519ba4ca7839354cbc731a42e
OPEN HERE:
https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments


In [27]:
import mlflow
import os

mlflow.set_tracking_uri("https://dagshub.com/lkhar21/ML-assignment2.mlflow")

os.environ["MLFLOW_TRACKING_USERNAME"] = "lkhar21"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "d2982c921140413d4212c1e34543c62d25a3c42c"

mlflow.set_experiment("FAST_XGBOOST")

2026/05/07 18:44:30 INFO mlflow.tracking.fluent: Experiment with name 'FAST_XGBOOST' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/49427e3450394f5184c84dec8b964f63', creation_time=1778179470493, experiment_id='5', last_update_time=1778179470493, lifecycle_stage='active', name='FAST_XGBOOST', tags={}, trace_location=None, workspace='default'>

In [28]:
with mlflow.start_run() as run:

    print("RUN STARTED:", run.info.run_id)

    # test log (must appear)
    mlflow.log_metric("test_metric", 1.0)

RUN STARTED: 490166b3d8bb4d40b18409b62b48a06b
🏃 View run angry-gull-793 at: https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments/5/runs/490166b3d8bb4d40b18409b62b48a06b
🧪 View experiment at: https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments/5


In [29]:
with mlflow.start_run() as run:

    model.fit(X_train, y_train)

    proba = model.predict_proba(X_val)[:, 1]

    mlflow.log_metric("auc", roc_auc_score(y_val, proba))

    print("Saved run:", run.info.run_id)

Saved run: 1778829d53cd42e6b7e9224e6c3cb90a
🏃 View run bouncy-sow-979 at: https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments/5/runs/1778829d53cd42e6b7e9224e6c3cb90a
🧪 View experiment at: https://dagshub.com/lkhar21/ML-assignment2.mlflow/#/experiments/5
